# Explore .inter Datasets (Beauty, Gowalla, Yelp)
Notebook này giúp bạn:
- Đọc dữ liệu từ file `.inter` (tab-separated)
- In sample rows
- Tính thống kê cơ bản cho từng dataset
- Vẽ biểu đồ phân phối tương tác user/item và top items

In [ ]:
from pathlib import Path
from collections import Counter
import csv
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('ggplot')

DATASETS = {
    'Beauty': Path('dataset/Beauty/Beauty.inter'),
    'Gowalla': Path('dataset/Gowalla/Gowalla.inter'),
    'Yelp': Path('dataset/Yelp/Yelp.inter'),
}

TOP_K = 15
SAMPLE_ROWS = 8

In [ ]:
def find_col(fieldnames, prefix):
    for c in fieldnames:
        if c == prefix or c.startswith(prefix + ':'):
            return c
    return None


def load_inter_dataset(file_path: Path):
    rows = []
    with file_path.open('r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        if not reader.fieldnames:
            raise ValueError(f'Missing header in {file_path}')

        user_col = find_col(reader.fieldnames, 'user_id')
        item_col = find_col(reader.fieldnames, 'item_id')
        ts_col = find_col(reader.fieldnames, 'timestamp')

        if user_col is None or item_col is None:
            raise ValueError(f'user_id/item_id columns not found in {file_path}')

        for row in reader:
            user = row.get(user_col)
            item = row.get(item_col)
            ts_raw = row.get(ts_col) if ts_col else None
            ts_val = None
            if ts_raw not in (None, ''):
                try:
                    ts_val = float(ts_raw)
                except ValueError:
                    ts_val = None

            rows.append({
                'user_id': user,
                'item_id': item,
                'timestamp': ts_val
            })

    return rows


def summarize_inter(rows):
    users = [r['user_id'] for r in rows]
    items = [r['item_id'] for r in rows]
    timestamps = [r['timestamp'] for r in rows if r['timestamp'] is not None]

    user_counts = Counter(users)
    item_counts = Counter(items)
    pair_counts = Counter((r['user_id'], r['item_id']) for r in rows)

    n_inter = len(rows)
    n_users = len(user_counts)
    n_items = len(item_counts)
    density = n_inter / (n_users * n_items) if n_users and n_items else 0.0

    return {
        'n_interactions': n_inter,
        'n_users': n_users,
        'n_items': n_items,
        'density': density,
        'duplicate_pairs': sum(v - 1 for v in pair_counts.values() if v > 1),
        'user_counts': user_counts,
        'item_counts': item_counts,
        'timestamp_min': min(timestamps) if timestamps else None,
        'timestamp_max': max(timestamps) if timestamps else None,
    }


def print_sample_rows(name, rows, n=8):
    print('=' * 90)
    print(f'Dataset: {name}')
    print('-' * 90)
    print(f'Sample rows (first {n}):')
    print('  user_id        item_id        timestamp')
    for r in rows[:n]:
        print(f"  {str(r['user_id']):<13} {str(r['item_id']):<13} {r['timestamp']}")

In [ ]:
all_stats = {}
all_rows = {}

for name, path in DATASETS.items():
    if not path.exists():
        print(f'[WARN] Missing file: {path}')
        continue

    rows = load_inter_dataset(path)
    stats = summarize_inter(rows)
    all_rows[name] = rows
    all_stats[name] = stats

    print_sample_rows(name, rows, n=SAMPLE_ROWS)
    print(f'Interactions      : {stats["n_interactions"]:,}')
    print(f'Unique users      : {stats["n_users"]:,}')
    print(f'Unique items      : {stats["n_items"]:,}')
    print(f'Density           : {stats["density"]:.8f}')
    print(f'Duplicate pairs   : {stats["duplicate_pairs"]:,}')
    if stats['timestamp_min'] is not None:
        print(f'Timestamp min/max : {stats["timestamp_min"]} / {stats["timestamp_max"]}')
    else:
        print('Timestamp min/max : not available')

In [ ]:
def plot_inter_distributions(name, stats, top_k=15):
    user_vals = np.array(list(stats['user_counts'].values()))
    item_vals = np.array(list(stats['item_counts'].values()))

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{name} - .inter Distribution Overview', fontsize=14, fontweight='bold')

    axes[0, 0].hist(user_vals, bins=40, color='#1D3557', alpha=0.9)
    axes[0, 0].set_title('Interactions per User')
    axes[0, 0].set_xlabel('Interactions')
    axes[0, 0].set_ylabel('Frequency')

    axes[0, 1].hist(item_vals, bins=40, color='#E76F51', alpha=0.9)
    axes[0, 1].set_title('Interactions per Item')
    axes[0, 1].set_xlabel('Interactions')
    axes[0, 1].set_ylabel('Frequency')

    top_users = stats['user_counts'].most_common(top_k)
    axes[1, 0].bar([str(u) for u, _ in top_users], [c for _, c in top_users], color='#2A9D8F')
    axes[1, 0].set_title(f'Top {top_k} Active Users')
    axes[1, 0].set_xlabel('User ID')
    axes[1, 0].set_ylabel('Interactions')
    axes[1, 0].tick_params(axis='x', rotation=75)

    top_items = stats['item_counts'].most_common(top_k)
    axes[1, 1].bar([str(i) for i, _ in top_items], [c for _, c in top_items], color='#F4A261')
    axes[1, 1].set_title(f'Top {top_k} Popular Items')
    axes[1, 1].set_xlabel('Item ID')
    axes[1, 1].set_ylabel('Interactions')
    axes[1, 1].tick_params(axis='x', rotation=75)

    plt.tight_layout()
    plt.show()


for name, stats in all_stats.items():
    plot_inter_distributions(name, stats, top_k=TOP_K)

In [ ]:
if all_stats:
    names = list(all_stats.keys())
    interactions = [all_stats[n]['n_interactions'] for n in names]
    users = [all_stats[n]['n_users'] for n in names]
    items = [all_stats[n]['n_items'] for n in names]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Comparison Across .inter Datasets', fontsize=14, fontweight='bold')

    axes[0].bar(names, interactions, color=['#264653', '#2A9D8F', '#E9C46A'])
    axes[0].set_title('Interactions')
    axes[0].tick_params(axis='x', rotation=20)

    axes[1].bar(names, users, color=['#E76F51', '#F4A261', '#8AB17D'])
    axes[1].set_title('Unique Users')
    axes[1].tick_params(axis='x', rotation=20)

    axes[2].bar(names, items, color=['#457B9D', '#A8DADC', '#1D3557'])
    axes[2].set_title('Unique Items')
    axes[2].tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.show()
else:
    print('No dataset found.')